## TP 2 Classification supervisée sur les documents du corpus (ANTOINE)

features: bow; tâche: attribution d'auteur ; 

Possibilité de faire l'implémentation de naïve bayes; utilisation de scikit learn; envisager peut-être aussi d'analyse d'erreur

### Énoncé

Sur quel corpus on travaille ?

Quelles features ? 

Quelle classification ? pourquoi  tant de haine ?

### Charger un jeu de données

In [ ]:
from datasets import load_dataset_builder

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

### Digression sur HuggingFace

Visitez : [Hugging Face](https://huggingface.co/)

In [ ]:
builder = load_dataset_builder(dataset_path)

print(f"Le jeu de données {dataset_path!r} contient {len(builder.info.splits)} sous-ensembles :")
for name, split in builder.info.splits.items():
    print(f"    {name:<20}{split.num_examples:<6,} documents")

Nous commençons par sélectionner l'un des 4 sous-ensembles de 500 document.

La variable `label_column` correspond à la colonne cible pour la classication pour chaque sous-ensemble

In [ ]:
# sous_ensemble, label_column = "auteur_genre", "auteur_genre"
# sous_ensemble, label_column = "auteur", "auteur"
# sous_ensemble, label_column = "date_publication", "date_publication"
sous_ensemble, label_column = "genre_litteraire", "genre_litteraire"

In [ ]:
from datasets import load_dataset

fichier = (
    f"hf://datasets/{dataset_path}/data/"
    f"{sous_ensemble}-00000-of-00001.parquet"
)

dataset = load_dataset(
    "parquet",
    data_files={"train": fichier},
    split="train"
)

dataset = dataset.to_list()

### Structure du dataset

In [25]:
print(f"Type dataset: {type(dataset)}")
print(f"Longueur dataset: {len(dataset)}")

doc_0 = dataset[0]
print(f"Type 1er document: {type(doc_0)}")
print(f"Clés d'un document: {doc_0.keys()}")

Type dataset: <class 'list'>
Longueur dataset: 500
Type 1er document: <class 'dict'>
Clés d'un document: dict_keys(['doc_id', 'date_publication', 'titre', 'auteur', 'auteur_genre', 'auteur_nom', 'auteur_prenom', 'genre_litteraire', 'texte'])


In [26]:
for key, value in doc_0.items():
    print(f"{key:<20} {type(value)}    {str(value)[:100]}")

doc_id               <class 'str'>    HDU-00056
date_publication     <class 'int'>    1802
titre                <class 'str'>    Armand et Angéla
auteur               <class 'str'>    Désirée de Castéra
auteur_genre         <class 'str'>    féminin
auteur_nom           <class 'str'>    Castéra
auteur_prenom        <class 'str'>    Désirée de
genre_litteraire     <class 'str'>    aventure
texte                <class 'str'>    Oubliez ce petit chagrin mon cher Henri, et ne vous occupez que du bonheur de posséder une femme cha


In [27]:
dataset[-1]["auteur"]

'Maurice Renard'

In [28]:
all_publication_date = [doc["date_publication"] for doc in dataset]
print("Min Year\t:", min(all_publication_date))
print("Max Year\t:", max(all_publication_date))
print("Mean Year\t:", int(sum(all_publication_date)/len(all_publication_date)))

Min Year	: 1802
Max Year	: 1946
Mean Year	: 1883


In [29]:
from collections import Counter
all_authors = [doc["auteur"] for doc in dataset]
Counter(all_authors).most_common(10)

[('Xavier de Montépin', 19),
 ('Paul Féval', 15),
 ('Jules Verne', 15),
 ('Jules Mary', 14),
 ('Gustave Aimard', 11),
 ('Adolphe Belot', 10),
 ('Alexis Bouvier', 10),
 ('Jules Lermina', 9),
 ('Pierre-Alexis Ponson du Terrail', 8),
 ('Alexandre Dumas', 7)]

In [30]:
all_authors_gender = [doc["auteur_genre"] for doc in dataset]
Counter(all_authors_gender).most_common()

[('masculin', 470), ('féminin', 30)]

In [32]:
all_genre = [doc["genre_litteraire"] for doc in dataset]
Counter(all_genre).most_common()

[('aventure', 250), ('detective', 250)]

### Prétraitement du texte

In [46]:
import re
def tokeniser(texte):
    texte = re.sub(r'[^\w\s]', ' ', texte) # Remplacer les signe de ponctuation par un espace
    texte = texte.lower() # convertir le texte en minuscules
    tokens = texte.split() # Découper le texte sur les espaces

    return tokens

In [47]:
exemple = """Longtemps, je me suis couché de bonne heure. 
Parfois, à peine ma bougie éteinte, mes yeux se fermaient si vite que je n’avais pas le temps de me dire : « Je m’endors. » 
Et, une demi-heure après, la pensée qu’il était temps de chercher le sommeil m’éveillait ; 
je voulais poser le volume que je croyais avoir encore dans les mains et souffler ma lumière ; 
je n’avais pas cessé en dormant de faire des réflexions sur ce que je venais de lire, mais ces réflexions avaient pris un tour un peu particulier ; 
il me semblait que j’étais moi-même ce dont parlait l’ouvrage : une église, un quatuor, la rivalité de François Ier et de Charles Quint."""

tokens = tokeniser(exemple)
print(tokens[:])

['longtemps', 'je', 'me', 'suis', 'couché', 'de', 'bonne', 'heure', 'parfois', 'à', 'peine', 'ma', 'bougie', 'éteinte', 'mes', 'yeux', 'se', 'fermaient', 'si', 'vite', 'que', 'je', 'n', 'avais', 'pas', 'le', 'temps', 'de', 'me', 'dire', 'je', 'm', 'endors', 'et', 'une', 'demi', 'heure', 'après', 'la', 'pensée', 'qu', 'il', 'était', 'temps', 'de', 'chercher', 'le', 'sommeil', 'm', 'éveillait', 'je', 'voulais', 'poser', 'le', 'volume', 'que', 'je', 'croyais', 'avoir', 'encore', 'dans', 'les', 'mains', 'et', 'souffler', 'ma', 'lumière', 'je', 'n', 'avais', 'pas', 'cessé', 'en', 'dormant', 'de', 'faire', 'des', 'réflexions', 'sur', 'ce', 'que', 'je', 'venais', 'de', 'lire', 'mais', 'ces', 'réflexions', 'avaient', 'pris', 'un', 'tour', 'un', 'peu', 'particulier', 'il', 'me', 'semblait', 'que', 'j', 'étais', 'moi', 'même', 'ce', 'dont', 'parlait', 'l', 'ouvrage', 'une', 'église', 'un', 'quatuor', 'la', 'rivalité', 'de', 'françois', 'ier', 'et', 'de', 'charles', 'quint']


In [49]:
from tqdm import tqdm

for document in tqdm(dataset):
    tokens = tokeniser(document["texte"])
    document["tokens"] = tokens

100%|██████████| 500/500 [00:17<00:00, 28.66it/s]


In [55]:
all_tokens = [token
              for document in dataset
              for token in document["tokens"]
              ]
vocab = sorted(set(all_tokens))
print(f"Taille du vocabulaire: {len(vocab):,} tokens")
print(f"{vocab[:10]} {vocab[-10:]}")

Taille du vocabulaire: 603,255 tokens
['0', '00', '000', '0001', '0002', '000bavarois', '000fr', '001', '0010', '0017'] ['χ', 'ψ', 'ϊ', 'ϊι', 'ϊν', 'ϋ', 'ύ', 'ύΐ', 'ώ', 'ӂ']


In [54]:
print(Counter(all_tokens).most_common(50))

[('de', 2662678), ('la', 1636516), ('le', 1527737), ('et', 1485505), ('à', 1292828), ('il', 1230284), ('l', 1121833), ('un', 905127), ('que', 870477), ('les', 859308), ('d', 814614), ('en', 766592), ('vous', 699031), ('je', 696728), ('une', 640377), ('qui', 591821), ('pas', 579452), ('est', 573055), ('ne', 552774), ('qu', 543451), ('se', 516600), ('s', 493678), ('elle', 491273), ('ce', 483081), ('des', 478781), ('dans', 461891), ('du', 447876), ('n', 425243), ('son', 413641), ('était', 410377), ('pour', 402294), ('lui', 377394), ('avait', 374477), ('a', 366054), ('au', 362766), ('sur', 309437), ('plus', 308409), ('mais', 303263), ('sa', 300871), ('c', 299670), ('m', 289063), ('on', 273648), ('par', 265979), ('avec', 261625), ('nous', 261405), ('dit', 241453), ('tout', 238930), ('j', 235281), ('ses', 234935), ('bien', 232264)]


#### Réduction du vocabulaire

In [53]:
token_count = dict(Counter(all_tokens))
token_unique = set([token for token, count in token_count.items() if count == 1])

print(f"{len(token_unique):,} tokens n'aparaissent qu'une seule fois dans l'ensemble du corpus. ({len(token_unique):,} / {len(vocab):,} = {len(token_unique)/len(vocab):.2%})")

354,013 tokens n'aparaissent qu'une seule fois dans l'ensemble du corpus. (354,013 / 603,255 = 58.68%)


In [57]:
filtered_vocab = set([token for token in vocab if token not in token_unique])
print(f"Taille du vocabulaire: {len(filtered_vocab):,} tokens")

Taille du vocabulaire: 249,242 tokens


Pour être utile à la classification, un token doit apparaître dans plusieurs documents. On cherche donc à savoir dans combien de document aparaissent chacun des tokens du vocabulaire.

In [64]:
vocab_documents = {token: {"occurrence": 0} for token in filtered_vocab}
for document in tqdm(dataset):
    tokens = set(document['tokens'])
    for token in tokens:
        if token in filtered_vocab:
            vocab_documents[token]["occurrence"] += 1
dataset_size = len(dataset)
for token in tqdm(vocab_documents):
    vocab_documents[token]["frequency"] = vocab_documents[token]["occurrence"]/dataset_size

100%|██████████| 249242/249242 [00:00<00:00, 1614213.15it/s]


In [121]:
min_documents_count = 2
max_documents_count = 400
# min_documents_ratio = 0.1
# max_documents_ratio = 0.8
min_token_length = 1

filtered_vocab = set([token for token in vocab_documents if vocab_documents[token]["occurrence"] >= min_documents_count
                                                    and vocab_documents[token]["occurrence"] <= max_documents_count
                                                    # and vocab_documents[token]["frequency"] >= min_documents_ratio
                                                    # and vocab_documents[token]["frequency"] <= max_documents_ratio
                                                    and len(token) > min_token_length
                 ])
print(f"Taille du vocabulaire: {len(filtered_vocab):,} tokens")

Taille du vocabulaire: 158,045 tokens


In [122]:
for document in tqdm(dataset):
    document["occurrence"] = dict(Counter([token for token in document["tokens"] if token in filtered_vocab]).most_common())
    filtered_tokens_count = sum(list(document["occurrence"].values()))
    document["frequence"] = {token:occurrence/filtered_tokens_count for token, occurrence in document["occurrence"].items()}

100%|██████████| 500/500 [00:07<00:00, 67.55it/s]


In [125]:
unique_labels = sorted(set(document[label_column] for document in dataset))

label2id = {
    genre: i
    for i, genre in enumerate(unique_labels)
}

id2label = {
    i: genre
    for genre, i in label2id.items()
}

print(label2id)
print(id2label)

{'aventure': 0, 'detective': 1}
{0: 'aventure', 1: 'detective'}


In [129]:
import pandas as pd

X_matrix = pd.DataFrame([document["frequence"] for document in dataset]).fillna(0)
X_matrix.head()
X_matrix.shape

(500, 158045)

In [130]:
X = X_matrix
y = pd.Series([
    label2id[document[label_column]]
    for document in dataset
])

groups = pd.Series([
    document["auteur"]
    for document in dataset
])

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import LeaveOneGroupOut
from tqdm import tqdm
import pandas as pd

X_matrix = pd.DataFrame(
    [document["frequence"] for document in dataset]
).fillna(0)

X = X_matrix

y = pd.Series([
    label2id[document[label_column]]
    for document in dataset
])

groups = pd.Series([
    document["auteur"]
    for document in dataset
])

logo = LeaveOneGroupOut()

results = []

for train_idx, test_idx in tqdm(
    logo.split(X, y, groups),
    total=groups.nunique(),
    desc="Leave-one-author-out"
):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    authors = groups.iloc[test_idx]

    model = RandomForestClassifier(
        max_depth=20,
        max_features="log2",
        min_samples_leaf=2,
        min_samples_split=2,
        n_estimators=200,
        n_jobs=4,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    for author, gold, pred in zip(authors, y_test, y_pred):
        results.append({
            "auteur": author,
            "gold_label": gold,
            "predicted_label": pred
        })

results_df = pd.DataFrame(results)

Leave-one-author-out:  55%|█████▌    | 137/249 [06:01<04:36,  2.47s/it]

In [ ]:
accuracy = accuracy_score(
    results["gold_label"],
    results["predicted_label"]
)

print(f"Accuracy: {accuracy:.3f}")